# Support triage crew: one store, every retrieval tool

A support agent reaches for the *right* kind of search per question — exact
keyword for an error code, semantic for a vague "how do I…", SQL for prices and
counts. With [`crewai-infino`](https://pypi.org/project/crewai-infino/) a CrewAI
crew gets all of them as tools over **one** [Infino](https://pypi.org/project/infino/)
store — vector, BM25, hybrid, and SQL over one copy of the data, with no second
vector database to keep in sync.

A researcher agent gathers the facts with those tools; a resolver agent writes
the reply. **The crew needs an LLM key** (the agents are tool-calling) — building
and searching the store is key-free. See the README.

In [1]:
import shutil
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # examples/ root, where _shared/ lives

import infino
import pyarrow as pa
from crewai import Agent, Crew, Process, Task
from crewai_infino import InfinoIndex

from _shared.crew import crew_llm
from _shared.embedding import DIM, METRIC, embed, embed_query
from _shared.loaders import load_amazon
from _shared.sql import query

DB_DIR = "./support_data"
TABLE = "support_kb"
shutil.rmtree(DB_DIR, ignore_errors=True)  # start fresh each run

db = infino.connect(DB_DIR)
llm = crew_llm()
print("LLM:", "on" if llm else "off (build + search still run; crew is skipped)")

LLM: on


## Build the support knowledge base

One `InfinoIndex` over a product catalog, with `price` / `rating` / `category`
promoted to scalar columns so SQL can filter on them. We add a few support
articles with known error codes and how-tos, so the demo ticket has clear
targets. `InfinoIndex.create` builds the BM25 and vector indexes; `add_texts`
embeds and appends in one call.

In [2]:
products = load_amazon(n=200)
index = InfinoIndex.create(
    db, TABLE,
    embed_documents=embed, embed_query=embed_query, dim=DIM, metric=METRIC,
    metadata_columns=[
        pa.field("price", pa.float64(), nullable=False),
        pa.field("rating", pa.float64(), nullable=False),
        pa.field("category", pa.large_utf8(), nullable=False),
    ],
)
index.add_texts(
    [p["text"] for p in products],
    metadatas=[{"price": p["price"], "rating": p["rating"], "category": p["category"]}
               for p in products],
)

# Seeded support articles with a known error code and a reset how-to.
ARTICLES = [
    "NovaSound Z9 earbuds: reset by holding both earbuds in the case for 10 seconds "
    "until the LED blinks white.",
    "Error E-507 on the NovaSound Z9 means a stale Bluetooth cache. Forget the device "
    "on your phone, then re-pair to clear it.",
    "HydroPeak 32oz bottle: the lid is dishwasher safe on the top rack; do not "
    "microwave the insulated body.",
]
index.add_texts(
    ARTICLES,
    metadatas=[{"price": 0.0, "rating": 0.0, "category": "Support"} for _ in ARTICLES],
    ids=[f"kb-{i}" for i in range(len(ARTICLES))],
)
print("indexed", query(db, f"SELECT COUNT(*) AS n FROM {TABLE}")["n"][0], "docs")

indexed 203 docs


## Four retrieval modes over the one store (key-free)

The agents' tools are thin wrappers over `index.searcher`. Each mode answers a
different kind of question over the *same* table — no LLM needed to show it:

- **keyword** (BM25, `mode="and"`) nails an exact error code,
- **semantic** (vector) matches a paraphrased intent,
- **SQL** counts and filters on the scalar columns.

In [3]:
s = index.searcher


def top(rows):
    return rows[0]["text"][:80] if rows else "(no hits)"


print("keyword  'E-507':       ", top(s.keyword("E-507", mode="and")))
print("semantic 'reset earbuds':", top(s.semantic("how do I reset my earbuds")))
print("sql      'under $30':    ",
      s.sql(f"SELECT COUNT(*) AS n FROM {TABLE} WHERE price > 0 AND price < 30")[0])

keyword  'E-507':        Error E-507 on the NovaSound Z9 means a stale Bluetooth cache. Forget the device
semantic 'reset earbuds': NovaSound Z9 earbuds: reset by holding both earbuds in the case for 10 seconds u
sql      'under $30':     {'n': 175}


## The triage crew

A **researcher** agent gathers facts with the Infino tools; a **resolver** agent
turns them into the customer reply. `index.as_tools()` hands the researcher all
four search tools at once — it reads their descriptions and picks per question.

In [4]:
def resolve(ticket: str) -> None:
    if llm is None:
        print(f"ticket: {ticket}\n(needs an LLM key — skipped)")
        return
    researcher = Agent(
        role="Support Researcher",
        goal="Gather the exact facts needed to resolve a customer ticket.",
        backstory="You search the knowledge base before answering — codes by keyword, "
                  "how-tos by meaning, prices and counts by SQL.",
        tools=index.as_tools(), llm=llm, verbose=False,
    )
    resolver = Agent(
        role="Support Resolver",
        goal="Write a concise, correct reply to the customer.",
        backstory="You turn the researched facts into a clear, friendly resolution.",
        llm=llm, verbose=False,
    )
    research = Task(
        description=f"Customer ticket: {ticket}\nFind every relevant fact from the "
                    "knowledge base needed to resolve it.",
        expected_output="The relevant facts: any error-code meaning and fix steps.",
        agent=researcher,
    )
    reply = Task(
        description="Write the customer-facing resolution for the ticket.",
        expected_output="A short, friendly reply that fixes the issue.",
        agent=resolver, context=[research],
    )
    crew = Crew(agents=[researcher, resolver], tasks=[research, reply],
                process=Process.sequential, verbose=False)
    # Notebooks run inside an event loop; run the sync crew in a worker thread.
    with ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(crew.kickoff).result()
    print(f"ticket: {ticket}\n\n{result}")


resolve("My NovaSound Z9 earbuds show error E-507 and won't reset. How do I fix it?")

ticket: My NovaSound Z9 earbuds show error E-507 and won't reset. How do I fix it?

Hi! Error **E-507** on the **NovaSound Z9** usually means there’s a stale Bluetooth cache.

Please try these steps:
1. On your phone, **forget/remove** the NovaSound Z9 from Bluetooth settings.
2. **Reset the earbuds** by placing both earbuds in the case, then **hold them for 10 seconds** until the **LED blinks white**.
3. Open your phone’s Bluetooth settings and **pair the earbuds again**.

This should clear the error and reconnect normally. If you’d like, I can also help walk you through the steps for your specific phone.


## Takeaway

One Infino store gave the crew keyword, semantic, hybrid, and SQL search as
agent tools — over one copy of the data, with no separate vector database. The
researcher picked the right mode per question; the resolver wrote the reply.